In [66]:
####################################
#ENVIRONMENT SETUP

In [67]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
from tqdm import tqdm

In [68]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [69]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [70]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [71]:
#IMPORT FUNCTIONS
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

#####

#Import StatisticalFunctions 
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
path=dir2+'Functions/'
sys.path.append(path)

import StatisticalFunctions
from StatisticalFunctions import * # import NumericalFunctions 

In [72]:
####################################
#LOADING CLASSES

In [73]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EulerianEntrainment",
                                dtype='float32')

=== CM1 Data Summary ===
 Simulation #:   4
 Resolution:     1km
 Time step:      3min
 Vertical levels:95
 Parcels:        20e6
 Data file:      /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Simulation_Four/cm1out_000001.nc
 Parcel file:    /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Simulation_Four/cm1out_pdata_1km_3min_20e6np.nc
 Time steps:     221

=== DataManager Summary ===
 inputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData
 outputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/EntrainmentCalculation
 inputDataDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData/Simulation_4_1km_3min_95nz/ModelData
 inputParcelDirec

In [74]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

Running timesteps from 0:7 



In [75]:
####################################
#DATA LOADING FUNCTIONS

In [76]:
def GetVelocityData(t):
    varNames = ['winterp','vinterp','uinterp']
    velDict = {varName: CallVariable(ModelData, DataManager, ModelData.timeStrings[t], 
                                  variableName=varName)\
                     for varName in varNames}
    return velDict

def GetActiveAir(t):
    varNames = ['rho','A_g','A_c']
    activeDict = {varName: CallVariable(ModelData, DataManager,
                                                    ModelData.timeStrings[t], 
                                                    variableName=varName)\
                              for varName in varNames}
    return activeDict

In [77]:
####################################
#COMPUTATION FUNCTIONS

In [78]:
#Entrainment Calculation Version One
def ComputeEntrainment_V1(activeDict_t1,activeDict_t0,velDict,
                       updraftType='c'):
    """
    Version One

    #Romps 2013 has extra recommendations not applied here yet, including
    #1 Since a cell can switch from inactive to active midway through a timestep. Use a fractional activity operator as defined in Romps 2013 Figure A1.
    #2 To ensure advected air is not counted as entrainment or detrainment, use rho*w located on cell face and interpolate A onto the cell face using a first-order upwind advection scheme and then calculate divergence. This ensures that activity flux comes from the upstream air mass. (this is doable, however model doesn't output rho on face and w on face was not selected for output)
    #3  To ensure pure boundary translation doesn't appear as alternating entrainment/detrainment at every timestep, accumulate activity source over each full period of adjacency. Sum the activity srouce over each timestep. (difficult since each cell has its own adjacency period)
    """

    rhoA_t1 = activeDict_t1['rho']*activeDict_t1[f'A_{updraftType}']
    rhoA_t0 = activeDict_t0['rho']*activeDict_t0[f'A_{updraftType}']
    drhoA_dt = (rhoA_t1 - rhoA_t0) / ModelData.dt

    w = velDict['winterp']; v = velDict['vinterp']; u = velDict['uinterp']
    rhoA_mid = 0.5 * (rhoA_t0 + rhoA_t1)
    divergence = (np.gradient(rhoA_mid * w, ModelData.zh*1e3, axis=0) +
                  np.gradient(rhoA_mid * v, ModelData.yh*1e3, axis=1) +
                  np.gradient(rhoA_mid * u, ModelData.xh*1e3, axis=2))

    activitySource = drhoA_dt + divergence
    entrainment=np.maximum(0,activitySource); detrainment=np.maximum(0,-activitySource)


    
    return entrainment,detrainment, rhoA_mid

def DivideMassFlux(activeDict_t1, E_g, D_g, E_c, D_c, velDict,
                   isDivideMean=True):

    rho = activeDict_t1['rho']
    A_g = activeDict_t1['A_g']
    A_c = activeDict_t1['A_c']
    
    
    if not isDivideMean:
        
        MF_g = rho * A_g * velDict['winterp']
        MF_c = rho * A_c * velDict['winterp']
    else:
        MF_g = np.nanmean(rho * A_g * velDict['winterp'], axis=(1,2))[:, np.newaxis, np.newaxis]  
        MF_c = np.nanmean(rho * A_c * velDict['winterp'], axis=(1,2))[:, np.newaxis, np.newaxis]

    E_g = NanDivide(E_g, MF_g)
    D_g = NanDivide(D_g, MF_g)
    E_c = NanDivide(E_c, MF_c)
    D_c = NanDivide(D_c, MF_c)
    
    return E_g, D_g, E_c, D_c

In [79]:
#Entrainment Calculation Version Two

def ComputeEntrainment_V2(activeDict_t1,activeDict_t0,velDict,
                       updraftType='c'):
    """
    Version Two
    
    additions from version 1 above:
    #2 To ensure advected air is not counted as entrainment or detrainment, use rho*w located on cell face and interpolate A onto the cell face using a first-order upwind advection scheme and then calculate divergence. This ensures that activity flux comes from the upstream air mass.
    """
    rhoA_t1 = activeDict_t1['rho']*activeDict_t1[f'A_{updraftType}']
    rhoA_t0 = activeDict_t0['rho']*activeDict_t0[f'A_{updraftType}']
    drhoA_dt = (rhoA_t1 - rhoA_t0) / ModelData.dt

    w = velDict['winterp']
    v = velDict['vinterp']
    u = velDict['uinterp']
    
    wFace = InterpolateToFaces(w, axis=0, zeroVerticalBoundary=True)
    vFace = InterpolateToFaces(v, axis=1)
    uFace = InterpolateToFaces(u, axis=2)

    rho_mid=0.5*(activeDict_t1['rho']+activeDict_t0['rho'])
    rhoFaceZ = InterpolateToFaces(rho_mid, axis=0, zeroVerticalBoundary=False)
    rhoFaceY = InterpolateToFaces(rho_mid, axis=1)
    rhoFaceX = InterpolateToFaces(rho_mid, axis=2)

    A_mid = 0.5 * (activeDict_t1[f'A_{updraftType}'] + activeDict_t0[f'A_{updraftType}'])
    AFaceZ = UpwindToFaces(A_mid, wFace, axis=0)
    AFaceY = UpwindToFaces(A_mid, vFace, axis=1)
    AFaceX = UpwindToFaces(A_mid, uFace, axis=2)
    
    fluxZ = rhoFaceZ * wFace * AFaceZ
    fluxY = rhoFaceY * vFace * AFaceY
    fluxX = rhoFaceX * uFace * AFaceX

    dz = np.diff(ModelData.zf * 1e3)[:, None, None]
    dy = np.diff(ModelData.yf * 1e3)[None, :, None]
    dx = np.diff(ModelData.xf * 1e3)[None, None, :]
    
    divZ = (fluxZ[1:, :, :] - fluxZ[:-1, :, :]) / dz
    divY = (fluxY - np.roll(fluxY, 1, axis=1)) / dy
    divX = (fluxX[:, :, 1:] - fluxX[:, :, :-1]) / dx
    
    divergence = divZ + divY + divX

    activitySource = drhoA_dt + divergence
    entrainment=np.maximum(0,activitySource); detrainment=np.maximum(0,-activitySource)

    return entrainment,detrainment,rho_mid*A_mid

def InterpolateToFaces(var3d, axis, zeroVerticalBoundary=False):
    if axis == 1:  # periodic y
        return 0.5 * (var3d + np.roll(var3d, -1, axis=1))

    if axis == 2:  # radiative x
        Nz, Ny, Nx = var3d.shape
        varFace = np.empty((Nz, Ny, Nx + 1), dtype=var3d.dtype)
        varFace[:, :, 1:-1] = 0.5 * (var3d[:, :, :-1] + var3d[:, :, 1:])
        varFace[:, :, 0] = var3d[:, :, 0]
        varFace[:, :, -1] = var3d[:, :, -1]
        return varFace

    if axis == 0:
        Nz, Ny, Nx = var3d.shape
        varFace = np.empty((Nz + 1, Ny, Nx), dtype=var3d.dtype)
        varFace[1:-1, :, :] = 0.5 * (var3d[:-1, :, :] + var3d[1:, :, :])

        if zeroVerticalBoundary:
            varFace[0, :, :] = 0.0
            varFace[-1, :, :] = 0.0
        else:
            varFace[0, :, :] = var3d[0, :, :]
            varFace[-1, :, :] = var3d[-1, :, :]

        return varFace

    raise ValueError("axis must be 0, 1, or 2")

def UpwindToFaces(A3d, velocityFace, axis):
    """
    Upwind interpolate a cell-centered activity field A3d
    onto cell faces using face-centered velocity.

    Parameters
    ----------
    A3d : ndarray
        Cell-centered activity field with shape (z, y, x)

    velocityFace : ndarray
        Face-centered velocity corresponding to the chosen axis

    axis : int
        0 = z faces
        1 = y faces (periodic)
        2 = x faces (radiative)

    Returns
    -------
    AFace : ndarray
        Activity interpolated to faces
    """

    # =========================
    # Z faces
    # =========================
    if axis == 0:

        Nz, Ny, Nx = A3d.shape
        AFace = np.empty((Nz + 1, Ny, Nx), dtype=A3d.dtype)

        AFace[1:-1, :, :] = np.where(
            velocityFace[1:-1, :, :] >= 0,
            A3d[:-1, :, :],   # upstream below
            A3d[1:, :, :]     # upstream above
        )

        # boundaries
        AFace[0, :, :] = A3d[0, :, :]
        AFace[-1, :, :] = A3d[-1, :, :]

        return AFace

    # =========================
    # Y faces (periodic)
    # =========================
    elif axis == 1:

        AForward = np.roll(A3d, -1, axis=1)

        AFace = np.where(
            velocityFace >= 0,
            A3d,       # upstream south
            AForward   # upstream north
        )

        return AFace

    # =========================
    # X faces (radiative)
    # =========================
    elif axis == 2:

        Nz, Ny, Nx = A3d.shape
        AFace = np.empty((Nz, Ny, Nx + 1), dtype=A3d.dtype)

        AFace[:, :, 1:-1] = np.where(
            velocityFace[:, :, 1:-1] >= 0,
            A3d[:, :, :-1],   # upstream west
            A3d[:, :, 1:]     # upstream east
        )

        # boundaries
        AFace[:, :, 0] = A3d[:, :, 0]
        AFace[:, :, -1] = A3d[:, :, -1]

        return AFace

    else:
        raise ValueError("axis must be 0, 1, or 2")

In [80]:
#Running Function
def RunCalculations(t):
    ####################################
    #DATA LOADING
    velDict = GetVelocityData(t=t)
    activeDict_t1 = GetActiveAir(t)
    activeDict_t0 = GetActiveAir(t-1)
    
    ####################################
    #COMPUTATION
    [E_g_eulerian, D_g_eulerian, rhoA_mid_g] = ComputeEntrainment_V2(activeDict_t1, activeDict_t0, velDict, updraftType='g')
    [E_c_eulerian, D_c_eulerian, rhoA_mid_c] = ComputeEntrainment_V2(activeDict_t1, activeDict_t0, velDict, updraftType='c')
    [E_g_eulerian_DMF, D_g_eulerian_DMF,
     E_c_eulerian_DMF, D_c_eulerian_DMF] = DivideMassFlux(activeDict_t1,
                                                          E_g_eulerian, D_g_eulerian,
                                                          E_c_eulerian, D_c_eulerian, velDict)
    
    outputDictionary = {
        'E_g_eulerian':     E_g_eulerian,
        'D_g_eulerian':     D_g_eulerian,
        'E_c_eulerian':     E_c_eulerian,
        'D_c_eulerian':     D_c_eulerian,
        'E_g_eulerian_DivideMassFlux': E_g_eulerian_DMF,
        'D_g_eulerian_DivideMassFlux': D_g_eulerian_DMF,
        'E_c_eulerian_DivideMassFlux': E_c_eulerian_DMF,
        'D_c_eulerian_DivideMassFlux': D_c_eulerian_DMF,
    }
    return outputDictionary

In [81]:
####################################
#RUNNING
running = True #keep true if running
# running = False

In [ ]:
if running:
    for t in tqdm(loop_elements):
        outputDictionary = RunCalculations(t=t)
        DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, 
                                       ModelData.timeStrings[t], outputDictionary)

In [ ]:
####################################
#TESTING

In [65]:
# # testing eulerian vs lagrangian comparison

# def PlotEntrainment(E, D, ax1, ax2, updraftType):
#     xcoord=ModelData.xh; zcoord=ModelData.zh
    
#     NET = E - D
#     b = np.nanmean(NET, axis=1)  # or whichever axis makes sense
#     vmax = np.nanpercentile(np.abs(b), 99)
#     levels = np.linspace(-vmax, vmax, 21)
    
#     cf = ax1.contourf(xcoord, zcoord, b, cmap='RdBu_r', levels=levels, extend='both')
#     cbar = plt.colorbar(cf, ax=ax1, 
#                         label=f'E_{updraftType} - D_{updraftType} (kg m$^{-3}$ s$^{-1}$)')
#     ax1.set_xlabel('x (km)'); 
#     ax1.set_ylabel('z (m)'); ax1.set_ylim(0,20)
    
#     ax2.plot(np.nanmean(E,   axis=(1, 2)), zcoord, 'blue',  label=f'E_{updraftType}')
#     ax2.plot(np.nanmean(D,   axis=(1, 2)), zcoord, 'red',   label=f'D_{updraftType}')
#     ax2.plot(np.nanmean(NET, axis=(1, 2)), zcoord, 'k',     
#              label=f'E_{updraftType} - D_{updraftType}',zorder=10)
#     ax2.axvline(0, color='k', lw=0.5)
#     ax2.set_xlabel('kg m$^{-3}$ s$^{-1}$'); ax2.legend()
#     ax2.set_ylim(0,20)

    
#     apply_scientific_notation([ax1,ax2])
#     apply_scientific_notation_colorbar([cbar])

# def PlotEntrainmentComparison(E_eul, D_eul, E_lag, D_lag, ax1, ax2, updraftType):
#     xcoord = ModelData.xh; zcoord = ModelData.zh
    
#     NET_eul = E_eul - D_eul
#     NET_lag = E_lag - D_lag
    
#     # contourf: lagrangian - eulerian
#     b = np.nanmean(NET_lag - NET_eul, axis=1)
#     vmax = np.nanpercentile(np.abs(b), 99)
#     levels = np.linspace(-vmax, vmax, 21)
#     cf = ax1.contourf(xcoord, zcoord, b, cmap='RdBu_r', levels=levels, extend='both')
#     cbar = plt.colorbar(cf, ax=ax1,
#                         label=f'(E-D)$_{{{updraftType},lag}}$ - (E-D)$_{{{updraftType},eul}}$ (kg m$^{{-3}}$ s$^{{-1}}$)')
#     ax1.set_xlabel('x (km)'); ax1.set_ylabel('z (m)'); ax1.set_ylim(0, 20)

#     # profiles: lagrangian=solid, eulerian=dashed
#     ax2.plot(np.nanmean(E_lag,   axis=(1,2)), zcoord, color='blue', ls='-',  label=f'E_{updraftType} lag')
#     ax2.plot(np.nanmean(D_lag,   axis=(1,2)), zcoord, color='red',  ls='-',  label=f'D_{updraftType} lag')
#     ax2.plot(np.nanmean(NET_lag, axis=(1,2)), zcoord, color='k',    ls='-',  label=f'E-D_{updraftType} lag', zorder=10)
#     ax2.plot(np.nanmean(E_eul,   axis=(1,2)), zcoord, color='blue', ls='--', label=f'E_{updraftType} eul')
#     ax2.plot(np.nanmean(D_eul,   axis=(1,2)), zcoord, color='red',  ls='--', label=f'D_{updraftType} eul')
#     ax2.plot(np.nanmean(NET_eul, axis=(1,2)), zcoord, color='k',    ls='--', label=f'E-D_{updraftType} eul')
#     ax2.axvline(0, color='k', lw=0.5)
#     ax2.set_xlabel('kg m$^{-3}$ s$^{-1}$'); ax2.legend(fontsize=7, ncol=2)
#     ax2.set_ylim(0, 20)

#     apply_scientific_notation([ax1, ax2])
#     apply_scientific_notation_colorbar([cbar])
    
# def ComparisonTest(t,
#                    isDivideMassFlux=False):
#     ###################################
#     # DATA LOADING
    

    
#     def GetLagrangianEntrainment(t,divideMassFluxString=False):
#         varNames = [f'{PROCESSED_string}Entrainment{divideMassFluxString}_g',f'{PROCESSED_string}Detrainment{divideMassFluxString}_g',
#                     f'{PROCESSED_string}Entrainment{divideMassFluxString}_c',f'{PROCESSED_string}Detrainment{divideMassFluxString}_c']
#         lagrangEntrainDict = {varName: CallVariable(ModelData, DataManager,
#                                                     ModelData.timeStrings[t], 
#                                                     variableName=varName)\
#                               for varName in varNames}
#         return lagrangEntrainDict
    
    
#     ###################################
#     # COMPUTATION
    
#     # Lagrangian Entrainment
#     PROCESSED_string=""
#     PROCESSED_string="PROCESSED_"
#     divideMassFluxString = "_DivideMassFlux" if isDivideMassFlux else ""
#     lagrangEntrainDict = GetLagrangianEntrainment(t=t,divideMassFluxString=divideMassFluxString)
#     E_g_lagrangian = lagrangEntrainDict[f'{PROCESSED_string}Entrainment{divideMassFluxString}_g']
#     D_g_lagrangian = -lagrangEntrainDict[f'{PROCESSED_string}Detrainment{divideMassFluxString}_g']
#     E_c_lagrangian = lagrangEntrainDict[f'{PROCESSED_string}Entrainment{divideMassFluxString}_c']
#     D_c_lagrangian = -lagrangEntrainDict[f'{PROCESSED_string}Detrainment{divideMassFluxString}_c']
    
#     # Eulerian Entrainment
#     velDict = GetVelocityData(t=t)
#     activeDict_t1 = GetActiveAir(t)
#     activeDict_t0 = GetActiveAir(t-1)
#     [E_g_eulerian,D_g_eulerian,rhoA_mid_g] = ComputeEntrainment_V2(activeDict_t1,activeDict_t0,velDict,
#                                      updraftType='g')
#     [E_c_eulerian,D_c_eulerian,rhoA_mid_c] = ComputeEntrainment_V2(activeDict_t1,activeDict_t0,velDict,
#                                      updraftType='c')
#     # divide by mass flux
#     if isDivideMassFlux:
#         [E_g_eulerian, D_g_eulerian, 
#          E_c_eulerian, D_c_eulerian] = DivideMassFlux(activeDict_t1,
#                                                               E_g_eulerian, D_g_eulerian, 
#                                                               E_c_eulerian, D_c_eulerian, 
#                                                               velDict)
    
#     ###################################
#     # PLOTTING
    
#     #Comparing Eulerian and Lagrangian Entrainment
    
#     # eulerian entrainment
#     fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
#     PlotEntrainment(E_g_eulerian, D_g_eulerian, ax1, ax2, updraftType='g')
#     PlotEntrainment(E_c_eulerian, D_c_eulerian, ax3, ax4, updraftType='c')
#     plt.tight_layout()
    
#     # lagrangian entrainment
#     fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
#     PlotEntrainment(E_g_lagrangian, D_g_lagrangian, ax1, ax2, updraftType='g')
#     PlotEntrainment(E_c_lagrangian, D_c_lagrangian, ax3, ax4, updraftType='c')
#     plt.tight_layout()
    
#     #lagrangian vs eulerian entrainment
#     fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
#     PlotEntrainmentComparison(E_g_eulerian, D_g_eulerian, E_g_lagrangian, D_g_lagrangian, ax1, ax2, updraftType='g')
#     PlotEntrainmentComparison(E_c_eulerian, D_c_eulerian, E_c_lagrangian, D_c_lagrangian, ax3, ax4, updraftType='c')
#     plt.tight_layout()

# ComparisonTest(t=210,
#                isDivideMassFlux=True)